# **CSCI 331 Project 2 - Sadia Sharmin**

## Step 1: Restore my partner's database

## Step 2: Create the Process.WorkflowSteps table

In [1]:
-- =============================================
-- Author: Sadia Sharmin
-- Procedure: Process.WorkflowSteps
-- Create date: 4/11/25
-- Description: Create the Process.WorkflowSteps table
-- =============================================

CREATE SCHEMA Process;
GO

CREATE TABLE Process.WorkflowSteps (
    WorkFlowStepKey INT NOT NULL, -- primary key
	WorkFlowStepDescription NVARCHAR (100) NOT NULL,
	WorkFlowStepTableRowCount INT NULL DEFAULT (0),
	StartingDateTime DATETIME2(7) NULL DEFAULT (SYSDATETIME ()),
	EndingDateTime DATETIME2(7) NULL DEFAULT (SYSDATETIME ()),
	ClassTime CHAR (5) NULL DEFAULT ('10:45'),
	UserAuthorizationKey INT NOT NULL 
);

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.052

## Step 3: Create a stored procedure called Process.usp\_TrackWorkFlows to add work flow to the table created previously

In [2]:
-- =============================================
-- Author: Sadia Sharmin
-- Procedure: [Process].[usp_TrackWorkFlows]
-- Create date: 4/11/25
-- Description: Tracks each step of the workflow
-- =============================================

CREATE OR ALTER PROCEDURE [Process].[usp_TrackWorkFlows]
    @StartTime DATETIME2,
    @WorkFlowDescription NVARCHAR(100),
    @WorkFlowStepTableRowCount INT,
    @UserAuthorizationKey INT
AS
BEGIN
    SET NOCOUNT ON;

    INSERT INTO Process.WorkflowSteps (
        WorkFlowStepDescription,
        WorkFlowStepTableRowCount,
        StartingDateTime,
        EndingDateTime,
        UserAuthorizationKey
    )
    VALUES (
        @WorkFlowDescription,
        @WorkFlowStepTableRowCount,
        @StartTime,
        SYSDATETIME(), 
        @UserAuthorizationKey
    );

    RETURN 0;
END;
GO

Commands completed successfully.

Total execution time: 00:00:00.018

## Step 4: Create stored procedure called Process.LoadDimOccupation to insert occupation information into our database from BIClass file and add tracking logic

In [3]:
-- =============================================
-- Author: Sadia Sharmin
-- Procedure: [Process].[LoadDimOccupation]
-- Create date: 4/11/25
-- Description: Loads occupation into DimOccupation and tracks flow
-- =============================================

USE [G10_2];
GO

DROP PROCEDURE IF EXISTS [Process].[LoadDimOccupation];
GO

CREATE PROCEDURE [Process].[LoadDimOccupation]
    @UserAuthorizationKey INT
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @StartTime DATETIME2 = SYSDATETIME();
    DECLARE @DateAdded DATETIME2 = SYSDATETIME();
    DECLARE @DateOfLastUpdate DATETIME2 = SYSDATETIME();

    -- insert OccupationName values from BIClass
    INSERT INTO [CH01-01-Dimension].[DimOccupation] (
        OccupationKey,
		OccupationName,
        UserAuthorizationKey,
        DateAdded,
        DateOfLastUpdate
    )
    SELECT 
		NEXT VALUE FOR PkSequence.OccupationSequenceObject,
        o.Occupation,
        @UserAuthorizationKey,
        @DateAdded,
        @DateOfLastUpdate
    FROM [BIClass].[CH01-01-Dimension].[DimOccupation] AS o
    WHERE o.Occupation IS NOT NULL AND NOT EXISTS (
          SELECT 1
          FROM [CH01-01-Dimension].[DimOccupation] d
          WHERE d.OccupationName = o.Occupation
      );

	--drop and recreate view to show loaded occupations
	IF OBJECT_ID('G10_2.uvw_DimOccupation', 'V') IS NOT NULL
        DROP VIEW G10_2.uvw_DimOccupation;
    
    EXEC('
        CREATE VIEW G10_2.uvw_DimOccupation AS
        SELECT *
        FROM [CH01-01-Dimension].[DimOccupation];
    ');

    -- tracking
    DECLARE @RowCount INT = (SELECT COUNT(*) FROM [CH01-01-Dimension].[DimOccupation]);
    EXEC Process.usp_TrackWorkFlows
        @StartTime,
        'Procedure: LoadDimOccupation → DimOccupation',
        @RowCount,
        @UserAuthorizationKey;
END;
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.036

## Step 5: Create stored procedure called Process.LoadDimProductCategory to insert information on the product category into our database from BIClass file and add tracking logic

In [4]:
-- =============================================
-- Author: Sadia Sharmin
-- Procedure: [Process].[LoadDimProductCategory]
-- Create date: 4/11/25
-- Description: Loads into DimProductCategory and tracks flow
-- =============================================

USE [G10_2]; 
GO

DROP PROCEDURE IF EXISTS [Process].[LoadDimProductCategory];
GO

CREATE PROCEDURE [Process].[LoadDimProductCategory]
    @UserAuthorizationKey INT
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @StartTime DATETIME2 = SYSDATETIME();
    DECLARE @DateAdded DATETIME2 = SYSDATETIME();
    DECLARE @DateOfLastUpdate DATETIME2 = SYSDATETIME();

    -- insert product categories from BIClass
    INSERT INTO [CH01-01-Dimension].[DimProductCategory] (
        ProductCategoryKey,
        ProductCategoryName,
        UserAuthorizationKey,
        DateAdded,
        DateOfLastUpdate
    )
    SELECT 
        NEXT VALUE FOR PkSequence.ProductCategorySequenceObject,
        p.ProductCategory,
        @UserAuthorizationKey,
        @DateAdded,
        @DateOfLastUpdate
    FROM [BIClass].[FileUpload].[ProductCategories] AS p
    WHERE p.ProductCategory IS NOT NULL
      AND NOT EXISTS (
          SELECT 1 
          FROM [CH01-01-Dimension].[DimProductCategory] AS d
          WHERE d.ProductCategoryName = p.ProductCategory

      );

    DECLARE @RowCount INT = @@ROWCOUNT;

    EXEC [Process].[usp_TrackWorkFlows]
        @StartTime,
        'Procedure: LoadDimProductCategory → DimProductCategory',
        @RowCount,
        @UserAuthorizationKey;
END;
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.024

## Step 6: Create stored procedure called Process.LoadDimProductSubcategory to insert  information on the product subcategory into our database from BIClass file and add tracking logic

In [5]:
-- =============================================
-- Author: Sadia Sharmin
-- Procedure: [Process].[LoadDimProductSubcategory]
-- Create date: 4/11/25
-- Description: Loads into DimProductSubcategory and tracks flow
-- =============================================

USE [G10_2];
GO

DROP PROCEDURE IF EXISTS [Process].[LoadDimProductSubcategory];
GO

CREATE PROCEDURE [Process].[LoadDimProductSubcategory]
    @UserAuthorizationKey INT
AS
BEGIN
    SET NOCOUNT ON;

    DECLARE @StartTime DATETIME2 = SYSDATETIME();
    DECLARE @DateAdded DATETIME2 = SYSDATETIME();
    DECLARE @DateOfLastUpdate DATETIME2 = SYSDATETIME();

    -- Insert distinct Subcategories and look up Category key
    INSERT INTO [CH01-01-Dimension].[DimProductSubcategory] (
        ProductSubcategoryKey,
        ProductCategoryKey,
        ProductSubcategoryName,
        UserAuthorizationKey,
        DateAdded,
        DateOfLastUpdate
    )
    SELECT 
        NEXT VALUE FOR PkSequence.ProductSubcategorySequenceObject,
        p.ProductCategoryKey,
        s.ProductSubcategory,
        @UserAuthorizationKey,
        @DateAdded,
        @DateOfLastUpdate
    FROM [BIClass].[FileUpload].[ProductSubcategories] AS s
    INNER JOIN [CH01-01-Dimension].[DimProductCategory] AS p
        ON p.ProductCategoryName = s.ProductSubcategory
    WHERE s.ProductSubcategory IS NOT NULL
      AND NOT EXISTS (
          SELECT 1 
          FROM [CH01-01-Dimension].[DimProductSubcategory] AS d
          WHERE d.ProductSubcategoryName = s.ProductSubcategory
      );

    DECLARE @RowCount INT = @@ROWCOUNT;

    EXEC [Process].[usp_TrackWorkFlows]
        @StartTime,
        'Procedure: LoadDimProductSubcategory → DimProductSubcategory',
        @RowCount,
        @UserAuthorizationKey;
END;
GO

Commands completed successfully.

Commands completed successfully.

Commands completed successfully.

Total execution time: 00:00:00.031

## Step 7: Back up the database and send to Pair 2